# Task 2 — Spam SMS / Email Classifier (“Phishing Simulator”)
TF-IDF + Logistic Regression gives a spam probability and explains influential words by multiplying TF-IDF values by learned class coefficients.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ART = Path("artifacts"); FIG = Path("figures"); EX = Path("examples")
for p in [ART, FIG, EX]:
    p.mkdir(exist_ok=True)
print("Folders ready:", ART, FIG, EX)

Folders ready: artifacts figures examples


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
LOCAL_FALLBACK = Path("sms.tsv")

def load_sms_dataset() -> pd.DataFrame:
    try:
        df = pd.read_csv(DATA_URL, sep="\t", header=None, names=["label", "text"])
        print(f"Loaded dataset from GitHub mirror: {len(df)} rows")
        return df
    except Exception as e:
        print(f"Could not fetch from URL ({e}); trying local fallback '{LOCAL_FALLBACK}'...")
        if LOCAL_FALLBACK.exists():
            df = pd.read_csv(LOCAL_FALLBACK, sep="\t", header=None, names=["label", "text"])
            print(f"Loaded dataset from local file: {len(df)} rows")
            return df
        raise RuntimeError(
            "Dataset could not be loaded from the URL or a local file. "
            f"Download the SMS Spam Collection dataset manually and save it as "
            f"'{LOCAL_FALLBACK}' (tab-separated, columns: label, text) next to this script."
        )

df = load_sms_dataset()
df = df.dropna(subset=["text", "label"])
df["target"] = (df["label"].astype(str).str.strip().str.lower() == "spam").astype(int)
print(df["target"].value_counts())

Loaded dataset from GitHub mirror: 5572 rows
target
0    4825
1     747
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

Xtr, Xte, ytr, yte = train_test_split(
    df.text, df.target, test_size=.20, random_state=42, stratify=df.target
)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
pipe.fit(Xtr, ytr)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(min_df=2, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

pred = pipe.predict(Xte)
prob = pipe.predict_proba(Xte)[:, 1]

metrics = {
    "accuracy": accuracy_score(yte, pred),
    "precision": precision_score(yte, pred),
    "recall": recall_score(yte, pred),
    "f1": f1_score(yte, pred),
}
print(metrics)
pd.DataFrame([metrics]).to_csv(ART / "metrics.csv", index=False)

ConfusionMatrixDisplay(
    confusion_matrix(yte, pred), display_labels=["ham", "spam"]
).plot(cmap="Blues")
plt.tight_layout()
plt.savefig(FIG / "confusion_matrix.png", dpi=160)
plt.close()

{'accuracy': 0.9757847533632287, 'precision': 0.8961038961038961, 'recall': 0.9261744966442953, 'f1': 0.9108910891089109}


In [ ]:
import joblib
joblib.dump(pipe, ART / "spam_classifier.joblib")

['artifacts/spam_classifier.joblib']

In [ ]:
def explain_spam(text, top_n=8):
    vec = pipe.named_steps["tfidf"]
    clf = pipe.named_steps["clf"]
    x = vec.transform([text])
    p = float(clf.predict_proba(x)[0, 1])
    names = np.array(vec.get_feature_names_out())
    idx = x.indices
    vals = x.data
    contributions = vals * clf.coef_[0][idx]
    order = np.argsort(contributions)[::-1]
    words = [(names[idx[i]], float(contributions[i])) for i in order if contributions[i] > 0][:top_n]
    return {"label": "SPAM" if p >= .5 else "HAM", "spam_probability": p, "trigger_keywords": words}

examples = [
    "Congratulations! You won a free prize. Click now to claim.",
    "Can we meet after class at 3 pm?",
    "URGENT: Your account is suspended. Verify your password immediately.",
    "Please send me the lecture notes.",
    "You have been selected for a cash reward. Call now!",
    "Your parcel will arrive tomorrow.",
    "Limited offer! Get 90% discount, act now.",
    "Mom said dinner is ready.",
    "Winner! Reply YES to collect your bonus.",
    "Reminder: project meeting moved to Friday.",
]

rows = []
for t in examples:
    r = explain_spam(t)
    rows.append({"text": t, **r})
    print(r, "\n", t, "\n")

pd.DataFrame(rows).to_csv(EX / "example_predictions.csv", index=False)

{'label': 'SPAM', 'spam_probability': 0.9757190903211271, 'trigger_keywords': [('claim', 1.3261788452467764), ('free', 1.1111856753045606), ('prize', 1.0499735204676524), ('won', 0.9840664831164426), ('congratulations', 0.28653846095663427), ('click', 0.24279271085107265)]} 
 Congratulations! You won a free prize. Click now to claim. 

{'label': 'HAM', 'spam_probability': 0.13513529158048435, 'trigger_keywords': [('pm', 0.19946705033049603)]} 
 Can we meet after class at 3 pm? 

{'label': 'SPAM', 'spam_probability': 0.6597659124846773, 'trigger_keywords': [('urgent', 0.8421670697719067), ('immediately', 0.37410315413277256), ('account', 0.3443168008847455), ('password', 0.21520284072615092), ('verify', 0.19371275505997188)]} 
 URGENT: Your account is suspended. Verify your password immediately. 

{'label': 'HAM', 'spam_probability': 0.2843785381338717, 'trigger_keywords': [('send', 0.5924588535301714)]} 
 Please send me the lecture notes. 

{'label': 'SPAM', 'spam_probability': 0.77430

In [ ]:
if __name__ == "__main__":
    import sys
    if sys.stdin.isatty():
        text = input("Type an SMS/email: ")
        print(explain_spam(text))
    else:
        print("(Skipping interactive prompt — no interactive stdin detected.)")

(Skipping interactive prompt — no interactive stdin detected.)
